# Chapter 3 — Coding Attention Mechanisms

Implementing self-attention: computing attention scores between a query token and all input tokens to build a context vector.

## Figure 3.8 — Attention scores for the context vector

![Figure 3.8: computing attention scores between the query x^(2) and all input elements as dot products](1.png)

*The overall goal is to illustrate the computation of the context vector $z^{(2)}$ using the second input element, $x^{(2)}$, as a query. This figure shows the first intermediate step: computing the attention scores $\omega$ between the query $x^{(2)}$ and all other input elements as a dot product. (The numbers are truncated to one digit after the decimal point to reduce visual clutter.)*

## Input embeddings

Each row is the 3-dimensional embedding of one token in the sentence *"Your journey starts with one step"* ($x^{(1)}$ through $x^{(6)}$).

In [12]:
import torch
import torch.nn as nn

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],  # step     (x^6)
    ]
)

query = inputs[1]
attention_scores = torch.zeros(inputs.shape[0])
for i in range(len(inputs)):
    attention_weights = torch.dot(query, inputs[i])
    attention_scores[i] = attention_weights

print(attention_scores) # -> then we normalize the attention scores  ( softmax)
# idea is to make add a learnable parameter to the attention weights

# shortcut

attnetion_scores = inputs @ inputs.T
print(attnetion_scores)
attention_weights = torch.softmax(attnetion_scores, dim=1)
print("normalized attention weights")
print(attention_weights)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])
normalized attention weights
tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [13]:
## Extend it with learnable weights !! yayy
## Using nn.Linear for the Q/K/V projections (better weight init, matches GPT-2 layout)
class SelfAttention(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)
        attn_scores = Q @ K.T
        # scale by sqrt(d_out): divide the scores, don't sqrt the scores
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)
        context_vector = attn_weights @ V
        return context_vector


sa = SelfAttention(d_in=3, d_out=2)
print(sa(inputs))


tensor([[-0.0327, -0.2101],
        [-0.0312, -0.2118],
        [-0.0313, -0.2118],
        [-0.0327, -0.2101],
        [-0.0329, -0.2099],
        [-0.0322, -0.2107]], grad_fn=<MmBackward0>)


### Causal attention, also known as masked attention

![](2.png)